In [1]:
import time
import pandas as pd
from datetime import datetime, timedelta
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

def scrape_senate_details():
    options = webdriver.ChromeOptions()
    # options.add_argument('--headless') # Uncomment to run in background
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    
    all_transactions = []

    try:
        driver.get("https://efdsearch.senate.gov/search/")
        wait = WebDriverWait(driver, 15)

        # 1. Bypass Agreement
        wait.until(EC.element_to_be_clickable((By.ID, "agree_statement"))).click()
        time.sleep(2) # Wait for search page to initialize

        # 2. Apply Filters (Senator & Periodic Transaction)
        print("Selecting 'Senator' filer type...")
        # Using a direct ID for the senator checkbox found in your HTML
        senator_check = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "senator_filer")))
        driver.execute_script("arguments[0].click();", senator_check)
        
        #driver.findElement(By.cssSelector("input[name='report_type'][value='7']")).click();

        # 4. Select 'Periodic Transaction Report'
        print("Selecting Periodic Transactions...")
        
        ptr_checkbox = wait.until(
            EC.element_to_be_clickable(
                (By.XPATH, "//input[@type='checkbox' and @name='report_type' and @value='11']")
            )
        )
        
        if not ptr_checkbox.is_selected():
            ptr_checkbox.click()
        # 3. Set Date (Past 20 Days)
        start_date = (datetime.now() - timedelta(days=20)).strftime("%m/%d/%Y")
        from_date = driver.find_element(By.ID, "fromDate")
        from_date.clear()
        from_date.send_keys(start_date)

        # 4. Search
        driver.find_element(By.CSS_SELECTOR, "button[type='submit']").click()
        
        # 5. Get all Report Links
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "#filedReports tbody tr")))
        time.sleep(2) # Buffer for DataTables
        
        report_rows = driver.find_elements(By.CSS_SELECTOR, "#filedReports tbody tr")
        reports_to_visit = []

        for row in report_rows:
            cells = row.find_elements(By.TAG_NAME, "td")
            if len(cells) >= 5 and "no matching" not in cells[0].text.lower():
                reports_to_visit.append({
                    'senator_name': f"{cells[0].text} {cells[1].text}",
                    'filing_date': cells[4].text,
                    'url': cells[3].find_element(By.TAG_NAME, "a").get_attribute("href")
                })

        print(f"Found {len(reports_to_visit)} reports. Starting deep scrape...")

        # 6. Visit each link and scrape the table
        for report in reports_to_visit:
            driver.get(report['url'])
            # Wait for the transaction table to appear
            try:
                wait.until(EC.presence_of_element_located((By.CLASS_NAME, "table-striped")))
                rows = driver.find_elements(By.CSS_SELECTOR, "table.table-striped tbody tr")
                
                for row in rows:
                    cols = row.find_elements(By.TAG_NAME, "td")
                    if len(cols) >= 9:
                        all_transactions.append({
                            "Senator": report['senator_name'],
                            "Filing Date": report['filing_date'],
                            "Transaction Date": cols[1].text.strip(),
                            "Owner": cols[2].text.strip(),
                            "Ticker": cols[3].text.strip(),
                            "Asset Name": cols[4].text.strip(),
                            "Asset Type": cols[5].text.strip(),
                            "Type": cols[6].text.strip(),
                            "Amount": cols[7].text.strip(),
                            "Comment": cols[8].text.strip()
                        })
            except:
                print(f"Could not parse transactions for {report['senator_name']}")
                continue

        # 7. Create DataFrame and Show
        df = pd.DataFrame(all_transactions)
        return df

    finally:
        driver.quit()

# Execute and display
final_df = scrape_senate_details()
print("\n--- SCRAPED TRANSACTION DATA ---")
print(final_df)

Selecting 'Senator' filer type...
Selecting Periodic Transactions...
Found 12 reports. Starting deep scrape...

--- SCRAPED TRANSACTION DATA ---
               Senator Filing Date Transaction Date   Owner Ticker  \
0         John Boozman  03/06/2026       02/19/2026   Joint    UNP   
1         John Boozman  03/06/2026       02/26/2026   Joint     PG   
2         John Boozman  03/06/2026       02/06/2026   Joint   JMBS   
3         John Boozman  03/06/2026       02/06/2026   Joint    SHY   
4         John Boozman  03/06/2026       02/06/2026   Joint    MBB   
..                 ...         ...              ...     ...    ...   
92  Sheldon Whitehouse  03/06/2026       02/23/2026    Self     HD   
93  Sheldon Whitehouse  03/06/2026       02/23/2026    Self    PEP   
94  Sheldon Whitehouse  03/06/2026       02/23/2026  Spouse     VZ   
95  Sheldon Whitehouse  03/06/2026       02/23/2026  Spouse     MA   
96  Sheldon Whitehouse  03/06/2026       02/23/2026  Spouse    PEP   

              

In [2]:
final_df

,Senator,Filing Date,Transaction Date,Owner,Ticker,Asset Name,Asset Type,Type,Amount,Comment
0,John Boozman,03/06/2026,02/19/2026,Joint,UNP,Union Pacific Corporation Common Stock,Stock,Sale (Partial),"$1,001 - $15,000",--
1,John Boozman,03/06/2026,02/26/2026,Joint,PG,Procter & Gamble Company (The) Common Stock,Stock,Sale (Partial),"$1,001 - $15,000",--
2,John Boozman,03/06/2026,02/06/2026,Joint,JMBS,Janus Henderson Mortgage-Backed Securities ETF,Stock,Sale (Partial),"$1,001 - $15,000",--
3,John Boozman,03/06/2026,02/06/2026,Joint,SHY,iShares 1-3 Year Treasury Bond ETF,Stock,Sale (Full),"$15,001 - $50,000",--
4,John Boozman,03/06/2026,02/06/2026,Joint,MBB,iShares MBS ETF,Stock,Sale (Partial),"$1,001 - $15,000",--
...,...,...,...,...,...,...,...,...,...,...
92,Sheldon Whitehouse,03/06/2026,02/23/2026,Self,HD,"Home Depot, Inc. (The) Common Stock",Stock,Sale (Full),"$1,001 - $15,000",--
93,Sheldon Whitehouse,03/06/2026,02/23/2026,Self,PEP,"PepsiCo, Inc. - Common Stock",Stock,Sale (Partial),"$1,001 - $15,000",--
94,Sheldon Whitehouse,03/06/2026,02/23/2026,Spouse,VZ,Verizon Communications Inc. Common Stock,Stock,Sale (Full),"$1,001 - $15,000",--
95,Sheldon Whitehouse,03/06/2026,02/23/2026,Spouse,MA,Mastercard Incorporated Common Stock,Stock,Sale (Partial),"$1,001 - $15,000",--


In [3]:
def df_to_csv(df):
    df.to_csv("senate_transactions_20_days.csv", index=False)
    print(f"Success! Saved {len(df)} records.")

In [4]:
df_to_csv(final_df)

Success! Saved 97 records.
